In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
import numpy as np

In [2]:
train_data = pd.read_csv(r'/Users/zulidobariya/Downloads/Train.csv')
test_data = pd.read_csv(r'/Users/zulidobariya/Desktop/Test.csv')

In [3]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8068 entries, 0 to 8067
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               8068 non-null   int64  
 1   Gender           8068 non-null   object 
 2   Ever_Married     7928 non-null   object 
 3   Age              8068 non-null   int64  
 4   Graduated        7990 non-null   object 
 5   Profession       7944 non-null   object 
 6   Work_Experience  7239 non-null   float64
 7   Spending_Score   8068 non-null   object 
 8   Family_Size      7733 non-null   float64
 9   Var_1            7992 non-null   object 
 10  Segmentation     8068 non-null   object 
dtypes: float64(2), int64(2), object(7)
memory usage: 693.5+ KB


In [4]:
num_cols = train_data.select_dtypes(include=['int64', 'float64']).columns
cat_cols = train_data.select_dtypes(include=['object']).columns

In [5]:
num_imputer = SimpleImputer(strategy='mean')
train_data[num_cols] = num_imputer.fit_transform(train_data[num_cols])

In [7]:
cat_imputer = SimpleImputer(strategy='most_frequent')
train_data[cat_cols] = cat_imputer.fit_transform(train_data[cat_cols])

In [8]:
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    train_data[col] = le.fit_transform(train_data[col])
    label_encoders[col] = le

In [9]:
scaler = StandardScaler()
train_data[num_cols] = scaler.fit_transform(train_data[num_cols])

In [10]:
X = train_data.drop(columns=['Segmentation']) 
y = train_data['Segmentation']

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

In [14]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
rf_val_preds = rf.predict(X_val)
rf_acc = accuracy_score(y_val, rf_val_preds)
print(f"Random Forest Accuracy: {rf_acc:.2f}")

Random Forest Accuracy: 0.51


In [17]:
et = ExtraTreesClassifier(random_state=42)
et.fit(X_train, y_train)
et_val_preds = et.predict(X_val)
et_acc = accuracy_score(y_val, et_val_preds)
print(f"ExtraTree Accuracy: {et_acc:.2f}")

ExtraTree Accuracy: 0.50


In [20]:
svm = SVC(probability=True, random_state=42)
svm.fit(X_train, y_train)
svm_val_preds = svm.predict(X_val)
svm_acc = accuracy_score(y_val, svm_val_preds)
print(f"SVM Accuracy: {svm_acc:.2f}")

SVM Accuracy: 0.51


In [22]:
ensemble = VotingClassifier(estimators=[('rf', rf), ('et', et), ('svm', svm)], voting='hard')
ensemble.fit(X_train, y_train)
ensemble_val_preds = ensemble.predict(X_val)
ensemble_acc = accuracy_score(y_val, ensemble_val_preds)
print(f"Ensemble Validation Accuracy: {ensemble_acc:.2f}")


Ensemble Validation Accuracy: 0.51


In [23]:
param_grid = {'C': [0.1, 1, 10], 'gamma': [1, 0.1, 0.01], 'kernel': ['rbf', 'linear']}
grid = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3)
grid.fit(X_train, y_train)
best_svm = grid.best_estimator_
best_svm.fit(X_train, y_train)


SVC(C=10, gamma=0.01, probability=True, random_state=42)

In [24]:
best_svm_test_preds = best_svm.predict(X_test)
best_svm_test_acc = accuracy_score(y_test, best_svm_test_preds)
print(f"Tuned SVM Test Accuracy: {best_svm_test_acc:.2f}")

Tuned SVM Test Accuracy: 0.53


In [25]:
ensemble_tuned = VotingClassifier(estimators=[('rf', rf), ('et', et), ('svm', best_svm)], voting='hard')
ensemble_tuned.fit(X_train, y_train)
ensemble_tuned_test_preds = ensemble_tuned.predict(X_test)
ensemble_tuned_test_acc = accuracy_score(y_test, ensemble_tuned_test_preds)
print(f"Enhanced Ensemble Test Accuracy: {ensemble_tuned_test_acc:.2f}")

Enhanced Ensemble Test Accuracy: 0.52


In [26]:
print(f"Final Test Accuracies:\nRandom Forest: {accuracy_score(y_test, rf.predict(X_test))}\nExtra Trees: {accuracy_score(y_test, et.predict(X_test))}\nSVM: {best_svm_test_acc}\nEnsemble: {ensemble_tuned_test_acc}")


Final Test Accuracies:
Random Forest: 0.5119735755573905
Extra Trees: 0.5045417010734929
SVM: 0.5276630883567299
Ensemble: 0.5243600330305532


In [27]:
feature_importances_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
feature_importances_et = pd.Series(et.feature_importances_, index=X.columns).sort_values(ascending=False)

In [28]:
print("\nRandom Forest Feature Importances:")
print(feature_importances_rf)


Random Forest Feature Importances:
ID                 0.239727
Age                0.239247
Profession         0.114741
Work_Experience    0.110461
Family_Size        0.089634
Var_1              0.059757
Spending_Score     0.053166
Graduated          0.034203
Gender             0.031137
Ever_Married       0.027927
dtype: float64


In [29]:
print("\nExtra Trees Feature Importances:")
print(feature_importances_et)


Extra Trees Feature Importances:
Age                0.223701
ID                 0.214782
Work_Experience    0.124671
Profession         0.119901
Family_Size        0.100081
Var_1              0.067290
Spending_Score     0.049149
Graduated          0.038559
Ever_Married       0.037449
Gender             0.024418
dtype: float64
